# Lesson 2 : Understanding the BERT Model and Its Role in LLMs and RAG

In this lesson, we will explore the BERT model (Bidirectional Encoder Representations from Transformers), a groundbreaking advancement in natural language processing (NLP). You have already learned about Large Language Models (LLMs) and Retrieval-Augmented Generation (RAG). Now, we will dive deeper into BERT, understand how it works, and see how it connects to LLMs and RAG systems. By the end of this lesson, you will have a clear understanding of BERT's architecture, its applications, and its role in modern NLP systems.

## Pre-training Objective: Masked Language Modeling (MLM)

In [1]:
! pip install transformers

In [2]:
from transformers import BertTokenizer, BertForMaskedLM
import torch

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForMaskedLM.from_pretrained('bert-base-uncased')
text = ("Napoleon revolutionised military organization"
"Napoleon has legacy"
"Legacy still lives"
)
rep = tokenizer(text, return_tensors = "pt")
print("Before Masking",rep.input_ids)
rand = torch.rand(rep.input_ids.shape)
mask_arr = (rand < 0.15) * (rep.input_ids != 101) * (rep.input_ids != 102)
selection = torch.flatten(mask_arr[0].nonzero()).tolist()
rep.input_ids[0, selection] = 103
after_masking = rep.input_ids
print("After Masking",after_masking)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Before Masking tensor([[  101,  8891,  4329,  5084,  2510,  3029,  2532, 15049,  2239,  2038,
          8027, 23115, 15719,  2145,  3268,   102]])
After Masking tensor([[  101,  8891,  4329,  5084,  2510,  3029,  2532, 15049,  2239,  2038,
          8027,   103,   103,  2145,   103,   102]])


# Hands-on Tutorial : Sentiment Analysis with BERT using Python

---



## Importing Dependencies

We will need the following libraries :

In [3]:
import torch
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification

## Downloading The IMDB Dataset

Use the link below to download a condensed subset of the IMDB Movie Reviews dataset—while the full collection includes 50,000 reviews, this version contains just 135.



In [4]:
df = pd.read_csv(
    'https://gist.githubusercontent.com/Mukilan-Krishnakumar/e998ecf27d11b84fe6225db11c239bc6/raw/74dbac2b992235e555df9a0a4e4d7271680e7e45/imdb_movie_reviews.csv'
)  # Read the IMDB reviews CSV directly from the given URL into a pandas DataFrame named df

df = df.drop('sentiment', axis=1)  # Remove the existing 'sentiment' column from df, keeping only the review text

## Model Building and Evaluation
We’ll leverage the bert-base-multilingual-uncased-sentiment model—a version of BERT that’s been fine-tuned specifically for assessing sentiment across multiple languages—to assign emotional valence to each review. To do this, we’ve implemented a helper function called sentiment_movie_score, which iterates over your dataset one row at a time, feeds each review text into the sentiment model, and interprets its output logits as a discrete rating on a 1-to-5 scale (where 1 indicates strongly negative sentiment and 5 indicates strongly positive sentiment). By encapsulating the tokenization, model inference, and score conversion inside this function, you can easily apply consistent, multilingual sentiment scoring to any collection of movie reviews with just a single function call.

In [5]:
tokenizer = BertTokenizer.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')
model = BertForSequenceClassification.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

In [6]:
def sentiment_score(movie_review):
	token = tokenizer.encode(movie_review, return_tensors = 'pt')
	result = model(token)
	return int(torch.argmax(result.logits))+1

In [7]:
df['sentiment'] = df['text'].apply(lambda x: sentiment_score(x[:512]))

In [8]:
df.head()

,text,sentiment
0,"My daughter liked it but I was aghast, that a ...",3
1,I... No words. No words can describe this. I w...,1
2,this film is basically a poor take on the old ...,2
3,"This is a terrible movie, and I'm not even sur...",1
4,First of all this movie is a piece of reality ...,4


## Visualising Results

In our experiment tracking setup, we start by initializing a Weights & Biases run with the project name “BERT_Sentiment_Analysis”, which organizes all related metrics and artifacts under a single, easily identifiable workspace. Next, we convert our pandas DataFrame of movie reviews and their predicted sentiment scores into a wandb.Table, allowing us to log the entire dataset as a structured artifact that can be browsed and filtered within the W&B UI. By calling wandb.log({"predictions": table}), we upload this table alongside any scalar metrics or visualizations we’ve collected. Finally, we call run.finish() to mark the end of the logging session—this closes out the run, ensures all data is properly synced, and makes the results available for later inspection and comparison.

In [9]:
!pip install wandb

In [10]:
import wandb
wandb.init(project="BERT_Sentiment_Analysis")
wandb.run.log({"Sentiment Analysis of IMDB Movie Reviews" : wandb.Table(dataframe=df)})
wandb.run.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tobilobaa (tobilobaa-developers-institute) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


By wrapping your DataFrame in a wandb.Table, you transform your raw predictions into a first-class artifact that can be browsed, filtered, and charted directly in the Weights & Biases UI. Once logged (e.g. via wandb.log({"predictions": table})), each column becomes a queryable field—so you can slice and dice by sentiment score, review length, or any other feature. The W&B dashboards will automatically generate summary statistics and visualizations (like histograms, scatter plots, or custom charts) based on your table’s schema, making it easy to spot trends or outliers at a glance. For a deeper dive into advanced querying, custom visualizations, and best practices, see the official wandb.Table documentation.



# **Challenge : Fine-Tune on a Split**

## Make a clean train/val split (80/20, stratified)

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split

# df must already exist from your previous step and have columns:
#   df["text"]  -> the review text
#   df["label"] -> integers 0..4  (map your 1..5 stars to 0..4 if needed)

# Example mapping if your labels are 1..5:
df["sentiment"] = df["sentiment"].astype(int) - 1

train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["sentiment"]
)

len(train_df), len(val_df)

(108, 27)

## Wrap into Hugging Face Datasets + tokenize

In [19]:
!pip install -q transformers datasets evaluate

from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df.reset_index(drop=True))
raw = DatasetDict({"train": train_ds, "validation": val_ds})

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

def tok_fn(batch):
    # Tokenize the text and include the 'sentiment' column as 'labels'
    tokenized_inputs = tokenizer(batch["text"], truncation=True, max_length=128, padding="max_length")
    tokenized_inputs["labels"] = batch["sentiment"]
    return tokenized_inputs

tok = raw.map(tok_fn, batched=True, remove_columns=["text", "sentiment"])

Map:   0%|          | 0/108 [00:00<?, ? examples/s]

Map:   0%|          | 0/27 [00:00<?, ? examples/s]

## Fine-tune the model (logs to your existing W&B project)

In [20]:
import wandb, numpy as np, evaluate
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

wandb.init(project="BERT_Sentiment_Analysis", job_type="finetune")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5  # nlptown is 5-class
)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy  = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall    = evaluate.load("recall")
f1        = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":  accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels, average="weighted")["precision"],
        "recall":    recall.compute(predictions=preds, references=labels, average="weighted")["recall"],
        "f1":        f1.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

args = TrainingArguments(
    output_dir="out",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",  # Set save_strategy to epoch
    logging_strategy="steps",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to=["wandb"],            # <-- sends metrics to W&B
    run_name="nlptown_135_finetune"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok["train"],
    eval_dataset=tok["validation"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics
)

trainer.train()
val_metrics = trainer.evaluate()
val_metrics

eval/accuracy,█▃▁█
eval/f1,█▃▁█
eval/loss,▁█▅▁
eval/precision,█▄▁█
eval/recall,█▃▁█
eval/runtime,▆█▁▃
eval/samples_per_second,▂▁█▆
eval/steps_per_second,▂▁█▆
train/epoch,▁▃▅████
train/global_step,▁▃▅▇███
+3,...


/tmp/ipython-input-1404126144.py:49: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.489224,0.888889,0.910053,0.888889,0.891580
2,0.598800,0.550508,0.814815,0.846561,0.814815,0.810858
3,0.312700,0.524829,0.777778,0.800705,0.777778,0.774756


{'eval_loss': 0.4892244040966034,
 'eval_accuracy': 0.8888888888888888,
 'eval_precision': 0.91005291005291,
 'eval_recall': 0.8888888888888888,
 'eval_f1': 0.891579613801836,
 'eval_runtime': 0.0908,
 'eval_samples_per_second': 297.384,
 'eval_steps_per_second': 11.014,
 'epoch': 3.0}